In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Load Saved Bayesian Project: LBCO, HRPT

This tutorial shows how to reopen the Bayesian project created in
`ed-21.py` and inspect the saved fit results without rerunning DREAM.

The project already contains posterior samples together with cached
posterior density, pair, and predictive data, so the plots below are
restored directly from disk.

## Import Library

In [2]:
from pathlib import Path

import easydiffraction as ed


# The ID 35 archive used below was saved before the
# switchable-category-owned-selectors refactor renamed several CIF
# tags. The helper below rewrites the archive in place so the tutorial
# can load it; it is intentionally narrow (ID 35 only, hrpt only,
# line-segment background only) and not a general legacy migration
# path. EasyDiffraction is in beta and does not ship legacy CIF
# shims, so saved projects in the old layout must be regenerated. The
# helper will be deleted once the upstream archive is republished
# under the current tag names.
def _normalize_id35_archive_for_tutorial(project_dir):
    """Rewrite the ID 35 archive's CIF tags for the current API."""
    project_path = Path(project_dir)

    replacements_by_file = {
        project_path / 'project.cif': {
            '_rendering.chart_engine': '_chart.type',
            '_rendering.table_engine': '_table.type',
        },
        project_path / 'analysis' / 'analysis.cif': {
            '_fitting.mode_type': '_fitting_mode.type',
            '_fitting.minimizer_type': '_minimizer.type',
        },
        project_path / 'experiments' / 'hrpt.cif': {
            '_calculation.calculator_type': '_calculator.type',
            '_peak.profile_type': '_peak.type',
        },
    }

    for file_path, replacements in replacements_by_file.items():
        text = file_path.read_text(encoding='utf-8')
        for old, new in replacements.items():
            text = text.replace(old, new)
        if file_path.name == 'hrpt.cif' and '_background.type' not in text:
            text = text.replace(
                '\nloop_\n_pd_background.id\n',
                '\n_background.type line-segment\nloop_\n_pd_background.id\n',
            )
        file_path.write_text(text, encoding='utf-8')

## Download Saved Project

The returned path points directly to the saved project directory with
the completed Bayesian fit and persisted posterior samples and plot
caches.

In [3]:
project_dir = ed.download_data(id=35, destination='projects')
_normalize_id35_archive_for_tutorial(project_dir)

Getting data...


Data #35: Bayesian Analysis: LBCO, HRPT


✅ Data #35 downloaded and extracted to
'/home/runner/work/diffraction-lib/diffraction-lib/projects/ed-35/lbco_hrpt_bayesian'


## Load the Saved Bayesian Project

Loading restores the persisted fit state, posterior samples, and plot
caches. No new fit is launched in this tutorial.

In [4]:
project = ed.Project.load(project_dir)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⚠️ Switching minimizer type removes these settings: max_iterations.                                                               


⚠️ Switching minimizer type adds these settings with defaults: burn_in_steps=600, initialization_method='latin_hypercube',        
   parallel_workers=0, population_size=4, random_seed=None, sampling_steps=3000, thinning_interval=1.                             


## Review the Saved Fit Summary

The fit summary reports the committed point estimate, sampler
settings, convergence diagnostics, and posterior parameter summaries
from the saved Bayesian run.

In [5]:
project.display.fit.results()

⚠️ Persisted posterior samples do not match restored posterior parameter names.                                                   


Bayesian fit results


⚠️ Overall status: completed with warnings


💬 Sampler status: DREAM sampling completed


🧪 Sampler: dream


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,start,best posterior sample,uncertainty,change
1,lbco,cell,,length_a,Å,3.8913,3.8913,0.0001,0.00 % ↓
2,hrpt,linked_phases,lbco,scale,,9.1330,9.1330,0.0290,0.00 % ↓
3,hrpt,peak,,broad_gauss_u,deg²,0.0817,0.0817,0.0066,0.00 % ↓
4,hrpt,peak,,broad_gauss_v,deg²,-0.1169,-0.1169,0.0047,0.00 % ↓
5,hrpt,instrument,,twotheta_offset,deg,0.6306,0.6306,0.0017,0.00 % ↓


📊 Posterior parameter summaries:


No posterior parameter summaries available.


## Show Correlations

The correlation matrix is restored from the saved project state.

In [6]:
project.display.fit.correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Inspect Posterior Densities and Pair Structure

The pair plot and one-dimensional posterior distributions now load
from the persisted caches generated when the Bayesian fit was saved.

In [7]:
project.display.posterior.pairs()

⚠️ Posterior samples are unavailable. Run a Bayesian fit first.                                                                   


In [8]:
project.display.posterior.distribution()

⚠️ Posterior samples are unavailable. Run a Bayesian fit first.                                                                   


⚠️ Posterior samples are unavailable. Run a Bayesian fit first.                                                                   


⚠️ Posterior samples are unavailable. Run a Bayesian fit first.                                                                   


⚠️ Posterior samples are unavailable. Run a Bayesian fit first.                                                                   


⚠️ Posterior samples are unavailable. Run a Bayesian fit first.                                                                   


## Plot Posterior Predictive Checks

The posterior predictive view reuses the cached predictive summary
stored in the project rather than recalculating it on first display.
It overlays the 95% credible interval propagated from the posterior
samples.

In [9]:
project.display.posterior.predictive(expt_name='hrpt')

A zoomed view is useful for checking the propagated uncertainty in a
narrow region of the diffraction pattern.

In [10]:
project.display.posterior.predictive(expt_name='hrpt', x_min=92, x_max=93)